## Video Game Hits from Pre-Release Metadata
### Final Project

### 1. Load the data

Loads `vgsales.csv` from the notebook folder.

In [84]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [85]:
RANDOM_STATE = 42
DATA_FILE = "vgsales.csv"

In [86]:
df_raw = pd.read_csv(DATA_FILE)

In [87]:
print(f"Loaded {DATA_FILE}: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Loaded vgsales.csv: 16,598 rows x 11 columns


In [88]:
df_raw.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


### Clean and Sanitize Dataset

Remove duplicates from vgsales.csv. Handle missing values and drop all columns that could leak information.

In [89]:
df_raw["hit_game"] = (df_raw["Global_Sales"] >= 1.0).astype(int)

In [90]:
drop_cols = ["Rank", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]
df = df_raw.drop(columns=drop_cols)

In [91]:
df = df.drop_duplicates()

In [92]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 16597 entries, 0 to 16597
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Name       16597 non-null  object 
 1   Platform   16597 non-null  object 
 2   Year       16327 non-null  float64
 3   Genre      16597 non-null  object 
 4   Publisher  16539 non-null  object 
 5   hit_game   16597 non-null  int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 907.6+ KB
None


In [93]:
print(df.isna().sum())

Name           0
Platform       0
Year         270
Genre          0
Publisher     58
hit_game       0
dtype: int64


### Data Inspection

Check the dataset size, column types, number of categories and sample entries.

In [94]:
print("Shape:", df.shape)

Shape: (16597, 6)


In [95]:
print("Unique publishers:", df["Publisher"].nunique())
print("Unique platforms:", df["Platform"].nunique())
print("Unique genres:", df["Genre"].nunique())

Unique publishers: 578
Unique platforms: 31
Unique genres: 12


In [96]:
print(df.describe(include="all").head(12))

                               Name Platform          Year   Genre  \
count                         16597    16597  16327.000000   16597   
unique                        11493       31           NaN      12   
top     Need for Speed: Most Wanted       DS           NaN  Action   
freq                             12     2163           NaN    3316   
mean                            NaN      NaN   2006.406443     NaN   
std                             NaN      NaN      5.828981     NaN   
min                             NaN      NaN   1980.000000     NaN   
25%                             NaN      NaN   2003.000000     NaN   
50%                             NaN      NaN   2007.000000     NaN   
75%                             NaN      NaN   2010.000000     NaN   
max                             NaN      NaN   2020.000000     NaN   

              Publisher      hit_game  
count             16539  16597.000000  
unique              578           NaN  
top     Electronic Arts           NaN  

In [97]:
print("hit_game value counts:")
print(df["hit_game"].value_counts())

hit_game value counts:
hit_game
0    14516
1     2081
Name: count, dtype: int64


### Leakage and Hygiene Control

Remove duplicates. Look for Year values like 0. Handle missing Year and Publisher values by imputing or categorizing them. Make sure all post-release sales and rank columns are removed.

In [98]:
df["Year"] = df["Year"].fillna(0)

In [99]:
df["Publisher"] = df["Publisher"].fillna("Unknown")

In [100]:
print("Year == 0 rows:", int((df["Year"] == 0).sum()))

Year == 0 rows: 270


### Exploratory Data Analysis

Look at the class balance of games are hits. Check hit rates across genres, platforms and publishers then examine distributions of features and correlation matrices.

In [101]:
print("Class balance:")
print(df["hit_game"].value_counts(normalize=True).mul(100).round(1))

Class balance:
hit_game
0    87.5
1    12.5
Name: proportion, dtype: float64


In [102]:
print("Hit rate by genre:")
print(df.groupby("Genre")["hit_game"].agg(["count", "mean"]).sort_values("count", ascending=False).head(10))

Hit rate by genre:
              count      mean
Genre                        
Action         3316  0.128468
Sports         2346  0.129156
Misc           1739  0.099482
Role-Playing   1488  0.136425
Shooter        1309  0.192513
Adventure      1286  0.032659
Racing         1249  0.145717
Platform        886  0.220090
Simulation      867  0.107266
Fighting        848  0.146226


In [103]:
print(df.corr(numeric_only=True))

              Year  hit_game
Year      1.000000  0.014515
hit_game  0.014515  1.000000


### Engineer Pre-Release Features

Create a release decade (1980s, 1990s). Identify console generation based on 31 platforms. Build publisher frequency features using training folds to avoid leakage.

In [104]:
df["release_decade"] = np.where(df["Year"] == 0, "Unknown", (df["Year"] // 10 * 10).astype(str) + "s")

In [105]:
platform_generation = {
    "2600": 2, "NES": 3, "SNES": 4, "N64": 5, "GC": 6, "Wii": 7, "GB": 3, "GBA": 5, "DS": 7, "3DS": 8,
    "PS": 4, "PS2": 6, "PS3": 7, "PS4": 8, "PSP": 7, "PSV": 8,
    "XB": 6, "360": 7, "XOne": 8, "PC": 5
}

In [106]:
df["console_generation"] = df["Platform"].map(platform_generation).fillna(0).astype(int)

In [107]:
print(df[["Year", "release_decade", "Platform", "console_generation"]].head())

     Year release_decade Platform  console_generation
0  2006.0        2000.0s      Wii                   7
1  1985.0        1980.0s      NES                   3
2  2008.0        2000.0s      Wii                   7
3  2009.0        2000.0s      Wii                   7
4  1996.0        1990.0s       GB                   3


### Train and Benchmark Classifiers

Use five algorithms: Logistic Regression, K-Nearest Neighbors, Decision Tree, Random Forest and Xgboost. Apply stratified cross validation and compare models using balanced class weights and SMOTE resampling.

In [108]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
%pip install --upgrade scikit-learn imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline

Note: you may need to restart the kernel to use updated packages.


In [109]:
feature_cols = ["Year", "Genre", "Publisher", "Platform", "release_decade", "console_generation"]
X = df[feature_cols]
y = df["hit_game"]

In [110]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [111]:
numeric_features = ["Year", "console_generation"]
categorical_features = ["Genre", "Publisher", "Platform", "release_decade"]

In [112]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

In [113]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [114]:
from imblearn.over_sampling import SMOTE

pipe_lr = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring="f1")
print(f"Logistic Regression: F1 = {scores_lr.mean():.3f} (+/- {scores_lr.std():.3f})")

C:\Users\James\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\James\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable

Logistic Regression: F1 = 0.406 (+/- 0.006)


C:\Users\James\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [115]:
pipe_knn = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", KNeighborsClassifier()),
])
scores_knn = cross_val_score(pipe_knn, X_train, y_train, cv=cv, scoring="f1")
print(f"K-Nearest Neighbors: F1 = {scores_knn.mean():.3f} (+/- {scores_knn.std():.3f})")

K-Nearest Neighbors: F1 = 0.309 (+/- 0.008)


In [116]:
pipe_dt = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
])
scores_dt = cross_val_score(pipe_dt, X_train, y_train, cv=cv, scoring="f1")
print(f"Decision Tree: F1 = {scores_dt.mean():.3f} (+/- {scores_dt.std():.3f})")

Decision Tree: F1 = 0.355 (+/- 0.033)


In [117]:
pipe_rf = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE)),
])
scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring="f1")
print(f"Random Forest: F1 = {scores_rf.mean():.3f} (+/- {scores_rf.std():.3f})")

Random Forest: F1 = 0.376 (+/- 0.033)


### Evaluate, Optimize and Explain

Compare models using accuracy, precision, recall, F1 score, Cohens kappa, ROC AUC and PR AUC. Optimize decision thresholds by testing values from 0.05 to 0.95 using assumed marketing cost per predicted hit and net return per positive. Compute SHAP values to explain model predictions.

In [118]:
rf = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE)),
])
rf.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](6,)","['Year','Genre','Publisher','Platform','release_decade', 'console_generation']"
n_features_in_,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3


In [119]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

In [120]:
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("F1:", round(f1_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("PR AUC:", round(average_precision_score(y_test, y_prob), 3))

Accuracy: 0.861
F1: 0.407
ROC AUC: 0.793
PR AUC: 0.387


In [121]:
thresholds = np.arange(0.05, 0.96, 0.05)

In [122]:
y_prob_col = y_prob.reshape(-1, 1)
thresholds_row = thresholds.reshape(1, -1)
preds = (y_prob_col >= thresholds_row).astype(int)
accs = (preds == y_test.to_numpy().reshape(-1, 1)).mean(axis=0)

In [123]:
print()
print(pd.DataFrame({"threshold": thresholds, "accuracy": accs.round(3)}))


    threshold  accuracy
0        0.05     0.634
1        0.10     0.720
2        0.15     0.769
3        0.20     0.798
4        0.25     0.823
5        0.30     0.835
6        0.35     0.844
7        0.40     0.850
8        0.45     0.857
9        0.50     0.861
10       0.55     0.865
11       0.60     0.868
12       0.65     0.870
13       0.70     0.874
14       0.75     0.877
15       0.80     0.878
16       0.85     0.881
17       0.90     0.880
18       0.95     0.879
